In [6]:
import pandas as pd
import numpy as np
import os
from sklearn.neighbors import KNeighborsClassifier
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

import os
import glob
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split   # dataset splitting
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import numpy as np

def print_label_counts(name, y):
    vals, counts = np.unique(y, return_counts=True)
    print(f"\n{name}")
    for v, c in zip(vals, counts):
        print(f"  Class {v}: {c}")

# Load datasets
natureDataTrain = np.load("/content/drive/MyDrive/trainDatasetNature.npz")
rugdDataTrain = np.load("/content/drive/MyDrive/trainDatasetRUGD.npz")
indoorDataTrain = np.load("/content/drive/MyDrive/trainDatasetIndoors.npz")

natureDataTest = np.load("/content/drive/MyDrive/testDatasetNature.npz")
rugdDataTest = np.load("/content/drive/MyDrive/testDatasetRUGD.npz")
indoorDataTest = np.load("/content/drive/MyDrive/testDatasetIndoors.npz")


def sample_balanced(X, y, n_per_class, seed=42):
    rng = np.random.default_rng(seed)

    class0_idx = np.where(y == 0)[0]
    class1_idx = np.where(y == 1)[0]

    class0_sample = rng.choice(class0_idx, size=n_per_class, replace=False)
    class1_sample = rng.choice(class1_idx, size=n_per_class, replace=False)

    keep_idx = np.concatenate([class0_sample, class1_sample])
    rng.shuffle(keep_idx)

    return X[keep_idx], y[keep_idx]

natureDataTrainX, natureDataTrainY = sample_balanced(
    natureDataTrain["x"],
    natureDataTrain["y"],
    n_per_class=750
)

natureDataTestX, natureDataTestY = sample_balanced(
    natureDataTest["x"],
    natureDataTest["y"],
    n_per_class=100
)


# Print counts
print_label_counts("Nature Train", natureDataTrainY)
print_label_counts("RUGD Train", rugdDataTrain["y"])
print_label_counts("Indoors Train", indoorDataTrain["y"])

print_label_counts("Nature Test", natureDataTestY)
print_label_counts("RUGD Test", rugdDataTest["y"])
print_label_counts("Indoors Test", indoorDataTest["y"])



trainX = np.concatenate((natureDataTrainX, rugdDataTrain["x"], indoorDataTrain["x"]))
trainY = np.concatenate((natureDataTrainY, rugdDataTrain["y"], indoorDataTrain["y"]))

testNatureX = natureDataTestX / 255.0
testNatureY = natureDataTestY

testRUGDX = rugdDataTest["x"] / 255.0
testRUGDY = rugdDataTest["y"]

testIndoorX = indoorDataTest["x"] / 255.0
testIndoorY = indoorDataTest["y"]

trainX = trainX.astype(np.float32) / 255.0

# shuffle train
perm = np.random.permutation(len(trainX))
trainX = trainX[perm]
trainY = trainY[perm]


Nature Train
  Class 0: 750
  Class 1: 750

RUGD Train
  Class 0: 952
  Class 1: 912

Indoors Train
  Class 0: 750
  Class 1: 750

Nature Test
  Class 0: 100
  Class 1: 100

RUGD Test
  Class 0: 6
  Class 1: 52

Indoors Test
  Class 0: 33
  Class 1: 227


In [8]:
# also must add an extra dimension
if trainX.ndim == 3:  # shape = (num_samples, 96, 96)
    print("addding dimesnsion")
    trainX = trainX[..., np.newaxis]  # add channel dim

if testNatureX.ndim == 3:  # shape = (num_samples, 96, 96)
    testNatureX = testNatureX[..., np.newaxis]  # add channel dim

if testRUGDX.ndim == 3:  # shape = (num_samples, 96, 96)
    testRUGDX = testRUGDX[..., np.newaxis]  # add channel dim

if testIndoorX.ndim == 3:  # shape = (num_samples, 96, 96)
    testIndoorX = testIndoorX[..., np.newaxis]  # add channel dim


randomIndexOrder = np.random.permutation(trainY.shape[0])

trainX = trainX[randomIndexOrder]
trainY = trainY[randomIndexOrder]


addding dimesnsion


In [9]:
print(trainX.shape)
print(trainY.shape)
print(testNatureX.shape)
print(testNatureY.shape)
print(testRUGDX.shape)
print(testRUGDY.shape)
print(testIndoorX.shape)
print(testIndoorY.shape)

(4864, 96, 96, 1)
(4864,)
(200, 96, 96, 1)
(200,)
(58, 96, 96, 1)
(58,)
(260, 96, 96, 1)
(260,)


In [10]:
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D
from tensorflow.keras.layers import Dense, Dropout, Activation


In [11]:
L2 = 1e-5

model = Sequential([
    Conv2D(16, (3,3), padding="same",
           kernel_regularizer=regularizers.l2(L2),
           input_shape=(96,96,1)),
    Activation("relu"),
    MaxPooling2D((2,2)),

    Conv2D(36, (3,3), padding="same",
           kernel_regularizer=regularizers.l2(L2)),
    Activation("relu"),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), padding="same",
           kernel_regularizer=regularizers.l2(L2)),
    Activation("relu"),

    GlobalAveragePooling2D(),

    Dense(36,
          activation="relu",
          kernel_regularizer=regularizers.l2(L2)),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3,
    decay_steps=1000,
    decay_rate=0.9
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# tensorflow is werid, so apparently you gotta compile it first? interesting
model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=['accuracy', 'AUC', 'Precision', 'Recall']
)

# history = model.fit(
#     trainingXSet, trainingYSet,
#     batch_size=16,
#     epochs=100
# )


model.fit(trainX, trainY, batch_size=32, epochs=80)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/80
152/152 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - AUC: 0.4934 - Precision: 0.4842 - Recall: 0.2861 - accuracy: 0.4949 - loss: 0.6945
Epoch 2/80
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC: 0.5413 - Precision: 0.5398 - Recall: 0.3624 - accuracy: 0.5306 - loss: 0.6920
Epoch 3/80
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC: 0.5782 - Precision: 0.5618 - Recall: 0.5141 - accuracy: 0.5602 - loss: 0.6852
Epoch 4/80
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC: 0.5975 - Precision: 0.5687 - Recall: 0.5883 - accuracy: 0.5746 - loss: 0.6803
Epoch 5/80
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC: 0.6063 - Precision: 0.5766 - Recall: 0.6086 - accuracy: 0.5843 - loss: 0.6764
Epoch 6/80
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC: 0.5997 - Precision: 0.5684 - Recall: 0.6045 - accuracy: 0.5763 - loss: 0.6785
Epoch 7/80
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - AUC: 0.6084 - Precision: 0.5709 - Recall: 0.6459 - accuracy: 0.5837 - loss: 0.6745
Epoch 8/80
152/152 ━━━━━━━━━━━━━━━━━━━━ 1

NameError: name 'testX' is not defined

In [13]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score

def runPredictions(model, theTestX, theTestY, name):
  predictions = model.predict(theTestX)

  # apparently this just works
  predicted_classes = (predictions > 0.5).astype(int)
  print(f"\n\nRUNNING FOR {name}:")
  confusionMatrix = confusion_matrix(theTestX, predicted_classes)
  print("Confusion Matrix:\n", confusionMatrix)

  accuracy = accuracy_score(theTestY, predicted_classes)
  print("Accuracy:\n", accuracy)

  recall = recall_score(theTestY, predicted_classes)
  print("Recall:\n", recall)

  f1Score = f1_score(theTestY, predicted_classes)
  print("F1 Score:\n", f1Score)

runPredictions(model, testNatureX, testNatureY, "Nature Dataset")
runPredictions(model, testRUGDX, testRUGDY, "RUGD Dataset")
runPredictions(model, testIndoorX, testIndoorY, "Indoor Dataset")


# scores are nothing to be proud of, but they are clearly learning something

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 


RUNNING FOR Nature Dataset:


ValueError: Classification metrics can't handle a mix of unknown and binary targets

In [ ]:
# its kinda weird, but it needs a dataset of representations to quantize to see what to scale immediate activation weights as
def representative_dataset():
    for i in range(500):  # take nearly all the samples from your training set
        # x_train[i] shape: (96, 96, 1), values between 0 and 1
        # Must add batch dimension
        # yield [trainX[i:i+1].astype('float32')]
        sample = trainX[i]
        sample = np.expand_dims(sample, axis=0).astype(np.float32)
        yield [sample]

def doConversion(model, fileName):

  # begin converting, not only to tflite but also doing quantization

  # specifies what we're gonna convert to
  converter = tf.lite.TFLiteConverter.from_keras_model(model)

  # give it the function to calculate middle activations
  converter.representative_dataset = representative_dataset

  converter.optimizations = [tf.lite.Optimize.DEFAULT]

  # Force int8 quantization for both weights and activations
  converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

  # Set input/output to uint8
  converter.inference_input_type = tf.int8
  converter.inference_output_type = tf.int8
  # actually make the quantized small model
  finalModel = converter.convert()

  # now, save it to a file
  with open(f"{fileName}.tflite", "wb") as f:
      f.write(finalModel)

  files.download(f"{fileName}.tflite")


In [ ]:
doConversion(model, "final_uint8_model")

In [ ]:
head_dense_layer   = model.layers[-1]          # Dense(1, sigmoid)
head_dense_weights = head_dense_layer.get_weights()  # [kernel, bias]

dropout_layer      = model.layers[-2]          # Dropout(0.25) — no weights

print("Dropout weights:")
print(dropout_layer.weights)
print("End dropout weights")

# Dense(1, sigmoid) — the removed head
kernel, bias = head_dense_weights
print("Head kernel shape:", kernel.shape)   # (32, 1)
print("Head bias shape:  ", bias.shape)     # (1,)
print("Kernel values:\n", kernel)
print("Bias value:  ", bias)

In [ ]:
headless_model = tf.keras.models.clone_model(model)
headless_model.set_weights(model.get_weights())

headless_model.pop()  # Dense(1, sigmoid)
headless_model.pop()  # Dropout — exists in Keras graph even though it's a no-op at inference

doConversion(headless_model, "final_uint8_model_headless")